# Notebook 05 — Cross-Encoder Reranking

**Phase 3 · Task group 115.** Goal: add a second-stage reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`) on top of the hybrid top-100 produced in Notebook 04, measure how much rank change it induces, and quantify the Precision@K gain.

### Why a cross-encoder
A bi-encoder (BGE-M3) sees the query and document *independently*, so it can't attend across them. A cross-encoder concatenates `[CLS] q [SEP] d [SEP]` and runs full attention over the pair — slower, but far more precise. The practical recipe is:

1. **Stage 1** (recall): hybrid top-100 candidates.
2. **Stage 2** (precision): cross-encoder scores each pair, top-5 wins.

### What this notebook ships
1. Two-stage pipeline demo on 3 queries with before/after rankings side-by-side.
2. Rank-change histogram — how aggressively does the reranker reorder?
3. Precision@K with vs. without reranking (paired, same candidate pool).
4. Persisted `reranking_precision.csv` for the paper's ablation table.


In [ ]:
import os, sys, json, time, pickle, warnings
from pathlib import Path
from collections import defaultdict

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR.parent / "2_src").exists() else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / "2_src"))

os.environ.setdefault("SCIRET_TIER", "tier1")
from config import get_config, BGE_M3_MODEL, CROSS_ENCODER_MODEL, DENSE_TOP_K, SPARSE_TOP_K, RRF_K, RERANK_TOP_K, SEED
CFG = get_config()
print(CFG.summary())

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); warnings.filterwarnings("ignore")


In [ ]:
chunks = pd.read_parquet(CFG.chunks_path)
text_lookup = dict(zip(chunks["chunk_id"], chunks["chunk_text"]))
title_lookup = dict(zip(chunks["chunk_id"], chunks["title"]))
print("chunks:", len(chunks))


## 1. Rebuild stage-1 retrievers

Both the BM25 pickle and the Chroma collection are artefacts of earlier notebooks — we just attach to them here.

In [ ]:
import chromadb, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

bge = SentenceTransformer(BGE_M3_MODEL, device=DEVICE)
client = chromadb.PersistentClient(path=str(CFG.chroma_dir))
col = client.get_collection(CFG.text_collection)

BM25_PATH = CFG.embeddings_dir / "bm25_index.pkl"
bm25, bm25_ids = pickle.loads(BM25_PATH.read_bytes())

def bm25_query(q, k=SPARSE_TOP_K):
    scores = bm25.get_scores([w for w in q.lower().split() if w.strip()])
    top = np.argsort(-scores)[:k]
    return [(bm25_ids[int(i)], float(scores[int(i)])) for i in top]

def dense_query(q, k=DENSE_TOP_K):
    qv = bge.encode([q], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k, include=["distances"])
    return list(zip(res["ids"][0], [1-float(d) for d in res["distances"][0]]))

def rrf(runs, k=RRF_K, top_k=100):
    s = defaultdict(float)
    for run in runs:
        for rank, (d,_) in enumerate(run, 1):
            s[d] += 1.0 / (k + rank)
    return sorted(s.items(), key=lambda x: x[1], reverse=True)[:top_k]

def hybrid(q, top_k=100):
    return rrf([dense_query(q), bm25_query(q)], top_k=top_k)

print("retrievers ok")


## 2. Load the cross-encoder

In [ ]:
reranker = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE)
# Sanity probe — a clear pair should score high, irrelevant pair should score low.
pairs = [
    ["What role does ACE2 play in SARS-CoV-2?",
     "The spike protein of SARS-CoV-2 binds ACE2 receptors on host cells to mediate entry."],
    ["What role does ACE2 play in SARS-CoV-2?",
     "The effects of weight training on muscle hypertrophy in young adults."],
]
print("reranker sanity:", reranker.predict(pairs, show_progress_bar=False).round(3).tolist())


## 3. Two-stage pipeline on 3 demo queries

In [ ]:
def two_stage(q, cand_k=100, top_k=RERANK_TOP_K):
    cand = hybrid(q, top_k=cand_k)
    ids = [d for d, _ in cand]
    scores = reranker.predict([[q, text_lookup.get(i,"")] for i in ids], show_progress_bar=False)
    order = np.argsort(-scores)
    reranked = [(ids[int(i)], float(scores[int(i)])) for i in order[:top_k]]
    # also return stage-1 top_k for direct comparison
    stage1 = cand[:top_k]
    return stage1, reranked, cand, scores

DEMO_QS = [
    "What imaging techniques were used to study COVID-19 lung damage?",
    "How effective are mRNA vaccines against the Delta variant?",
    "What role does IL-6 play in COVID-19 cytokine storm?",
]

for q in DEMO_QS:
    s1, s2, _, _ = two_stage(q)
    print("Q:", q)
    print("  Stage 1 (hybrid):")
    for i, (d, s) in enumerate(s1, 1):
        print(f"    {i}. {d}  rrf={s:.4f}  | {title_lookup.get(d,'')[:80]}")
    print("  Stage 2 (reranked):")
    for i, (d, s) in enumerate(s2, 1):
        print(f"    {i}. {d}  ce={s:.3f}  | {title_lookup.get(d,'')[:80]}")
    print()


## 4. Rank-change analysis

For each query, compare the rank of each stage-1 candidate before vs. after reranking. Mean `|rank_after - rank_before|` quantifies how much reordering the reranker does.

In [ ]:
QUERIES = [
    "What imaging techniques were used to study COVID-19 lung damage?",
    "How effective are mRNA vaccines against the Delta variant?",
    "What role does the ACE2 receptor play in SARS-CoV-2 infection?",
    "Which antiviral drugs showed efficacy against SARS-CoV-2 in vitro?",
    "What are the cardiac complications of COVID-19?",
    "How is long COVID defined and what are its symptoms?",
    "Describe neutralising antibody response after BNT162b2 vaccination.",
    "What is the role of IL-6 in COVID-19 cytokine storm?",
    "How did remdesivir perform in randomised clinical trials?",
    "What are risk factors for severe COVID-19 in older adults?",
    "How accurate are RT-PCR tests for SARS-CoV-2 detection?",
    "Which CT imaging features are typical of COVID-19 pneumonia?",
    "How does SARS-CoV-2 evolve — what are key variant lineages?",
    "What neurological symptoms are associated with COVID-19?",
    "What is the impact of COVID-19 on pregnancy outcomes?",
    "How effective are face masks for community transmission?",
    "How did the pandemic affect mental health?",
    "What is the effectiveness of convalescent plasma for COVID-19?",
    "Describe the hyperinflammatory syndrome in paediatric COVID-19.",
    "How does SARS-CoV-2 spread between humans indoors?",
]
print(len(QUERIES), "queries")


In [ ]:
change_rows, score_rows = [], []
for q in QUERIES:
    cand = hybrid(q, top_k=50)
    ids = [d for d,_ in cand]
    ce = reranker.predict([[q, text_lookup.get(i,"")] for i in ids], show_progress_bar=False)
    order = np.argsort(-ce)
    # rank before (0-indexed in `cand`) vs rank after
    before = {d:i for i,d in enumerate(ids)}
    after = {ids[int(i)]:r for r,i in enumerate(order)}
    for d in ids:
        change_rows.append({"q": q, "doc": d, "before": before[d],
                            "after": after[d], "delta": after[d]-before[d]})

chg = pd.DataFrame(change_rows)
summary = chg.groupby("q")["delta"].agg(lambda s: np.mean(np.abs(s))).describe().round(2)
print("mean |Δrank| across queries:", chg["delta"].abs().mean().round(2))
summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(chg["delta"], bins=np.arange(-50,51,2), color="#8b5cf6", edgecolor="white")
ax.axvline(0, color="red", lw=1)
ax.set_title("Reranker-induced position change (Δrank = after − before)")
ax.set_xlabel("Δrank")
ax.set_ylabel("# documents")
fig.tight_layout()
fig.savefig(CFG.results_dir / "reranker_rank_change.png", dpi=140)
plt.show()


## 5. Precision@K — with vs without reranking

We use the same silver-relevance rule as Notebook 04 (top of the hybrid run corroborated by cross-encoder), but keep the evaluation fair by fixing the candidate pool and only changing the *ordering* step.

In [ ]:
def silver(q, top_n=10, thresh=0.0):
    cand = hybrid(q, top_k=20)
    ids = [d for d,_ in cand]
    ce = reranker.predict([[q, text_lookup.get(i,"")] for i in ids], show_progress_bar=False)
    pos = [i for i,s in zip(ids, ce) if s >= thresh]
    return set(pos[:top_n]) if pos else set(ids[:5])

def precision_at_k(rel, retrieved, k):
    if k == 0: return 0.0
    return len([r for r in retrieved[:k] if r in rel]) / k

KS = [1, 3, 5, 10]
rows = []
for q in QUERIES:
    rel = silver(q)
    cand = hybrid(q, top_k=50)
    ids = [d for d,_ in cand]
    # without reranking -> keep hybrid order
    s1_run = ids
    # with reranking
    ce = reranker.predict([[q, text_lookup.get(i,"")] for i in ids], show_progress_bar=False)
    s2_run = [ids[int(i)] for i in np.argsort(-ce)]
    for k in KS:
        rows.append({"query": q, "k": k,
                     "p_no_rerank": precision_at_k(rel, s1_run, k),
                     "p_rerank":    precision_at_k(rel, s2_run, k)})

prec_df = pd.DataFrame(rows)
agg = prec_df.groupby("k")[["p_no_rerank","p_rerank"]].mean().round(3)
agg


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(agg.index))
ax.bar(x-0.2, agg["p_no_rerank"], 0.4, label="Stage 1 only (hybrid)", color="#64748b")
ax.bar(x+0.2, agg["p_rerank"],    0.4, label="+ Cross-encoder rerank", color="#22c55e")
ax.set_xticks(x); ax.set_xticklabels([f"P@{k}" for k in agg.index])
ax.set_ylim(0, 1.05); ax.set_ylabel("Precision")
ax.set_title(f"Precision@K — with vs without reranking (tier={CFG.tier})")
ax.legend()
fig.tight_layout()
fig.savefig(CFG.results_dir / "reranking_precision.png", dpi=140)
agg.to_csv(CFG.results_dir / "reranking_precision.csv")
plt.show()


In [ ]:
# Also save the per-query table (used later for a paired t-test in Notebook 09).
prec_df.to_csv(CFG.results_dir / "reranking_precision_per_query.csv", index=False)
delta = (prec_df.pivot(index="query", columns="k", values="p_rerank")
       - prec_df.pivot(index="query", columns="k", values="p_no_rerank"))
print("mean Δprecision per K:")
delta.mean().round(3)


---
**Outputs**
* `4_results/<tier>/reranker_rank_change.png`
* `4_results/<tier>/reranking_precision.{png,csv}`
* `4_results/<tier>/reranking_precision_per_query.csv`

**Next:** Notebook 06 — PDF figure extraction (start of the multimodal branch).
